# AR Rehabilitation RL Controller - Phase 1 Base Model Training
## Unilateral Spatial Neglect (USN) Unity Telemetry Adaptation

### Phase 1 Overview
This notebook executes **Phase 1: Pre-training the Base RL Model** (PPO) using simulated patient sessions on `UnityARRehabEnv`.
- **State Space**: 15-Dimensional Normalized Observation Vector (gaze offset, reaction time, hit rate, fatigue, neglect side).
- **Action Space**: 4 Dynamic Trial Actions (`speed`, `eccentricity_deg`, `distance_m`, `time_limit_s`).
- **Output**: Pre-trained `checkpoints/unity_ppo_model.zip` ready for Phase 2 Continuous Live Adaptation in Unity.

In [ ]:
# Install dependencies if needed on Kaggle or Google Colab
# !pip install gymnasium stable-baselines3 matplotlib plotly pandas numpy torch flask

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, DQN, A2C

from unity_schema import UnitySession, UnityTrial, DifficultyAtTrial
from unity_env import UnityARRehabEnv
from unity_dataset import session_to_observation, predict_next_difficulty
from utils import set_seeds

set_seeds(42)
print("Phase 1 Setup & Dependencies Successfully Loaded!")

### 1. Test Environment & Sample Unity JSON Parsing

In [ ]:
sample_json_path = "sample_unity_session.json"
with open(sample_json_path, "r", encoding="utf-8") as f:
    sample_json = f.read()

session = UnitySession.from_json_str(sample_json)
print(f"Loaded Session ID: {session.session_id} for Patient: {session.patient_id}")
print(f"Total Recorded Trials: {len(session.trials)}")

# Extract 15-dim state observation
obs = session_to_observation(session)
print("15-Dim Extracted Observation Vector:", obs)

### 2. Phase 1: Pre-training the Base PPO Model (50,000 Steps)
We train the primary PPO agent on `UnityARRehabEnv` to establish safe baseline knowledge before live deployment.

In [ ]:
from train_unity import train_unity_agent

# Pre-train Base PPO Model
base_ppo_model, ppo_logs = train_unity_agent(algo_name="PPO", total_timesteps=50000, seed=42)
print("Phase 1 Base Model Pre-training Complete!")

### 3. Generate Next Trial Recommendations
We test predicting the optimal 4-action difficulty settings for the sample patient session.

In [ ]:
recommendation = predict_next_difficulty(sample_json, model=base_ppo_model)
import json
print(json.dumps(recommendation, indent=2))